<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-05/NB05_symbol_grounding.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 05: Symbol Grounding and Meaning

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-05/lab.html>.

## Overview

How can the symbols of a system mean anything to the system itself? This week examines the symbol grounding problem, the distributional view of meaning that underlies language models, the argument that form alone cannot yield meaning, and replies that locate meaning in conceptual roles or learned relations [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to state the symbol grounding problem and Harnad's proposed solution, to compute distributional word representations and interpret their similarities, to reconstruct the octopus argument of Bender and Koller, and to evaluate replies based on conceptual roles and vector grounding.

## The grounding problem

Harnad asked how the meanings of symbols in a formal system can be intrinsic to the system rather than parasitic on the meanings in the heads of its users [1]. He illustrated the problem with a dictionary-go-round: Someone who knows no Chinese and tries to learn it from a Chinese-Chinese dictionary will pass from one meaningless string to another without end, because every definition consists of further undefined symbols. The Chinese Room of Week 4 can be read as a case of the same problem.

Harnad proposed a hybrid solution. Elementary symbols are grounded in two kinds of non-symbolic representation: iconic representations, which are analogues of sensory projections, and categorical representations, which are feature detectors that pick out the invariant properties of categories. Higher-level symbols are then composed from grounded ones, so that a new word such as zebra can inherit meaning from horse and stripes. Grounding, on this view, requires a causal connection between symbols and the things they are about.

## Meaning from distribution

A different tradition locates meaning in patterns of use. Harris proposed that words that occur in similar contexts tend to have similar meanings [2]. Modern vector semantics turns the idea into computation: Each word is represented by the contexts in which it occurs, often after weighting co-occurrence counts by positive pointwise mutual information and reducing their dimension. Figure 5.1 shows the result for a toy corpus: Animals, vehicles and fruits separate into regions, although the program has never seen an animal, a vehicle or a fruit. Large language models are distributional learners on an enormous scale, which is why the grounding problem has returned with new urgency.

![Figure 5.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-05/figures/w05_fig1.png)

*Figure 5.1. Toy corpus words positioned by their co-occurrence statistics alone. The categories separate even though the program has no access to animals, vehicles or fruits [2].*

## Form and meaning

Bender and Koller argued that a system trained only on linguistic form cannot learn meaning, understood as the relation between form and something outside language, such as communicative intent [3]. Their thought experiment features a hyperintelligent octopus that taps an undersea cable between two people stranded on separate islands. The octopus learns the statistical patterns of their messages well enough to impersonate one of them. When the other asks for help to build a device from local materials or to escape a bear, the octopus fails, because it has never had access to what the words are about.

Others resist the conclusion. Piantadosi and Hill argued that meaning need not depend on reference, since in conceptual role theories the meaning of a concept is fixed by its relations to other concepts, and language models may capture such relations [4]. Mollo and Millière distinguished several senses of grounding and argued that the relevant sense for language models is referential grounding, whose conditions do not require multimodal input or embodiment [5]. The debate turns on what meaning is, not only on what the models do.

## Grounding as a matter of degree

The positions can be compared by asking what connects symbols to the world in each case. For Harnad, the connection runs through perception. For distributional semantics, it runs through the texts that people wrote about the world, so it is indirect. For conceptual role theories, relations among concepts may carry part of the meaning even without reference. The lab makes the contrast concrete: One part computes similarities from co-occurrence alone, and another follows chains of definitions until they either loop or reach words marked as grounded in perception. The notebook then tests whether a mapping learned from a few grounded words transfers to others.

> **Pause and reflect.** When a language model describes the taste of a lemon, is it referring to lemons? What would have to be added for the answer to change?

# Hands-on lab

The notebook builds distributional word vectors from a generated corpus, inspects their neighbourhoods and then grounds a few words in simple perceptual features to test whether the grounding transfers to words that were never grounded [1, 2, 5].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A generated corpus and distributional vectors

In [ ]:
rng = np.random.default_rng(SEED)
CATS = {
    "animal": (["cat", "dog", "horse", "cow", "sheep", "goat"], ["eats", "runs", "sleeps", "barn", "farm", "fur", "tail"]),
    "fruit": (["apple", "banana", "cherry", "lemon", "grape", "pear"], ["ripe", "sweet", "juice", "tree", "peel", "seeds"]),
    "vehicle": (["car", "bus", "truck", "train", "bike", "boat"], ["drives", "road", "fuel", "wheels", "engine", "driver"]),
}
GENERIC = ["big", "small", "old", "new", "near", "today"]
sentences = []
for _ in range(3000):
    cat = rng.choice(list(CATS))
    words, ctx = CATS[cat]
    sentences.append([rng.choice(words)] + list(rng.choice(ctx, 2, replace=False)) + [rng.choice(GENERIC)])
vocab = sorted({w for s in sentences for w in s})
ix = {w: i for i, w in enumerate(vocab)}
C = np.zeros((len(vocab), len(vocab)))
for s in sentences:
    for a in s:
        for b in s:
            if a != b:
                C[ix[a], ix[b]] += 1
tot = C.sum(); pw = C.sum(1, keepdims=True) / tot
with np.errstate(divide="ignore"):
    PPMI = np.maximum(np.log((C / tot) / (pw @ pw.T)), 0)
PPMI[~np.isfinite(PPMI)] = 0
U, S, _ = np.linalg.svd(PPMI)
EMB = U[:, :6] * S[:6]
print(len(vocab), "words, embedding dimension", EMB.shape[1])

### Your turn, exercise 1

Write `cosine(u, v)` and use it to list the three nearest neighbours of `apple` among the other target words.

In [ ]:
def cosine(u, v):
    # Your code here
    return None

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _cosine_reference(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

In [ ]:
check("Exercise 1", cosine(EMB[ix["apple"]], EMB[ix["pear"]]), _cosine_reference(EMB[ix["apple"]], EMB[ix["pear"]]))
targets = [w for ws, _ in CATS.values() for w in ws]
sims = sorted(((w, _cosine_reference(EMB[ix["apple"]], EMB[ix[w]])) for w in targets if w != "apple"), key=lambda t: -t[1])
print("nearest to apple:", [(w, round(s, 3)) for w, s in sims[:3]])

## 2. Grounding a few words in perception

Four perceptual features are attached to the target words: animate, edible, has wheels and typical size. A linear map from the distributional vectors to these features is learned from a few grounded words and applied to the rest.

In [ ]:
from sklearn.linear_model import Ridge
FEAT = {}
for w in CATS["animal"][0]: FEAT[w] = [1, 0, 0, 0.6]
for w in CATS["fruit"][0]:  FEAT[w] = [0, 1, 0, 0.1]
for w in CATS["vehicle"][0]: FEAT[w] = [0, 0, 1, 0.8]
FEAT["bike"][3] = 0.4; FEAT["boat"][2] = 0; FEAT["cat"][3] = 0.3
grounded = ["cat", "horse", "apple", "lemon", "car", "train"]
held_out = [w for w in targets if w not in grounded]
mapper = Ridge(alpha=0.1).fit(EMB[[ix[w] for w in grounded]], np.array([FEAT[w] for w in grounded]))
pred = mapper.predict(EMB[[ix[w] for w in held_out]])
for w, p in zip(held_out, pred):
    print(f"{w:7s} predicted animate {p[0]:+.2f}  edible {p[1]:+.2f}  wheels {p[2]:+.2f}")

### Your turn, exercise 2

Compute the accuracy of the predicted `edible` feature on the held-out words, counting a prediction above 0.5 as edible. Then discuss in three sentences what this transfer does and does not show about grounding.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _edible_accuracy_reference():
    return float(np.mean([(p[1] > 0.5) == bool(FEAT[w][1]) for w, p in zip(held_out, pred)]))

In [ ]:
edible_accuracy = None  # your computation
check("Exercise 2", edible_accuracy, _edible_accuracy_reference())

## Visual exploration: Meaning from distribution

Twelve words receive vectors from co-occurrence counts alone, weighted by positive pointwise mutual information and reduced to two dimensions. The words cluster by category, although the system never perceives an animal, a food or a tool.

In [ ]:
rng = np.random.default_rng(SEED)
groups = {"animals": ["cat", "dog", "horse", "sheep"], "foods": ["bread", "apple", "rice", "cheese"], "tools": ["hammer", "saw", "drill", "wrench"]}
contexts = {"animals": ["feed", "pet", "barn", "fur", "run"], "foods": ["eat", "cook", "taste", "kitchen", "fresh"], "tools": ["use", "workshop", "repair", "metal", "handle"]}
words = [w for ws in groups.values() for w in ws]
cats = [g for g, ws in groups.items() for _ in ws]
ctx_vocab = sorted({c for cs in contexts.values() for c in cs})
M = np.zeros((len(words), len(ctx_vocab)))
for _ in range(4000):
    i = rng.integers(len(words))
    c = contexts[cats[i]][rng.integers(5)] if rng.random() < 0.8 else ctx_vocab[rng.integers(len(ctx_vocab))]
    M[i, ctx_vocab.index(c)] += 1
P = M / M.sum()
ppmi = np.maximum(np.log((P + 1e-12) / (P.sum(1, keepdims=True) * P.sum(0, keepdims=True) + 1e-12)), 0)
U, S, _ = np.linalg.svd(ppmi, full_matrices=False)
xy = U[:, :2] * S[:2]
colours = {"animals": "tab:orange", "foods": "tab:green", "tools": "tab:blue"}
for (x, yv), word, c in zip(xy, words, cats):
    plt.scatter(x, yv, color=colours[c]); plt.annotate(word, (x, yv), fontsize=9)
plt.xlabel("dimension 1"); plt.ylabel("dimension 2"); plt.title("Word vectors from co-occurrence alone"); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-05/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which position on meaning from this week do you find most plausible for language models, and what evidence would change your mind?
2. The notebook transferred features from grounded to ungrounded words. Is that genuine grounding or a sophisticated dictionary-go-round?
3. Choose a technical term from your field. How did you come to understand it: through definitions, through use, through perception or through practice?

## Weekly task and submission

Write about 500 words comparing Harnad's grounding requirement with the conceptual role view of Piantadosi and Hill, applied to a language model answering questions in your field. Use the notebook results as evidence where appropriate and attach the completed notebook.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Do multimodal models ground their symbols?.** Examine whether models trained on images and text meet the conditions of the grounding requirement, contrasting Harnad's view with the vector grounding debate [1, 3, 5].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Harnad, S. (1990). The symbol grounding problem. *Physica D: Nonlinear Phenomena*, 42(1-3), 335-346. <https://doi.org/10.1016/0167-2789(90)90087-6>

[2] Harris, Z. S. (1954). Distributional structure. *Word*, 10(2-3), 146-162. <https://doi.org/10.1080/00437956.1954.11659520>

[3] Bender, E. M., & Koller, A. (2020). Climbing towards NLU: On meaning, form, and understanding in the age of data. In *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics* (pp. 5185-5198). <https://doi.org/10.18653/v1/2020.acl-main.463>

[4] Piantadosi, S. T., & Hill, F. (2022). Meaning without reference in large language models. arXiv preprint arXiv:2208.02957. <https://arxiv.org/abs/2208.02957>

[5] Mollo, D. C., & Millière, R. (2023). The vector grounding problem. arXiv preprint arXiv:2304.01481. <https://arxiv.org/abs/2304.01481>